In [ ]:
import model
import epidemic_simulation
import survey_design
import random
import numpy as np
import scipy.stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.cm as cm
import arviz
import cmdstanpy
from cmdstanpy import CmdStanModel

In [ ]:
random.seed(123)
np.random.seed(123)

In [ ]:
f_infectious, f_recovered, delay_inf, delay_recov = epidemic_simulation.delays()
transmission_rate = epidemic_simulation.transmission_rate

In [ ]:
inference_model = CmdStanModel(stan_file='model_rw2.stan', cpp_options={'STAN_THREADS':'true'})

spacings = [21, 42, 84, 168]
sample_sizes = [1000, 2000, 4000, 8000, 16000, 32000, 64000]

num_repeats = 10

dfs = []
all_scores = []
all_fits = []
all_expanded_scores = []
all_scores_energy = []
all_coverages = []
all_widths = []

In [ ]:
for _ in range(num_repeats):

    m = model.AgentModel(1000000, 50)

    full_spacings = np.repeat(spacings, len(sample_sizes))
    full_sample_sizes = np.tile(sample_sizes, len(spacings))

    surveys = survey_design.make_surveys(m, full_spacings, full_sample_sizes, full_spacings, full_sample_sizes, duration=7)
    df = survey_design.run_simulations(m, surveys, transmission_rate, f_infectious, f_recovered)

    fits = survey_design.run_inference(df, surveys, inference_model, delay_inf, delay_recov, 'combined', init_mcmc=800)
    scores, expanded_scores, scores_energy, coverage, width = survey_design.evaluate_fits(df, fits)

    dfs.append(df)
    all_scores.append(scores)
    all_fits.append(fits)
    all_expanded_scores.append(expanded_scores)

    all_scores_energy.append(scores_energy)

    all_coverages.append(coverage)
    all_widths.append(width)


In [ ]:
all_scores = np.asarray(all_scores)
avg_scores = np.median(all_scores, axis=0)

all_scores_energy = np.asarray(all_scores_energy)
avg_scores_energy = np.median(all_scores_energy, axis=0)

In [ ]:
fig = plt.figure(figsize=(5, 3.5))
ax = fig.add_subplot(1, 1, 1)


# Compute contours of total survey tests conducted

dense_spacing = np.arange(16, 220, 5)
dense_ss = np.arange(700, 90000, 100)

X, Y = np.meshgrid(dense_spacing, dense_ss)

effort = np.zeros(X.shape)
for i, spacing in enumerate(dense_spacing):
    for j, ss in enumerate(dense_ss):
        surveys = survey_design.make_surveys(m, [spacing], [ss], [spacing], [ss], duration=7)
        effort[j, i] = surveys[0].sample_size * len(surveys[0].start_dates) + surveys[1].sample_size * len(surveys[1].start_dates)

levels = [10000, 20000, 40000, 80000, 160000, 320000, 640000, 1280000]

true_levels = []
for l in levels:
    nearest = min(effort.flatten(), key=lambda x:abs(x-l))
    true_levels.append(nearest)

cs = ax.contour(X, Y, effort, true_levels, colors='tab:red', zorder=-10,)
ax.clabel(cs, cs.levels, fontsize=10)


# Plot quality at each considered survey design

cmap = mpl.colormaps['Greys_r']

highest_crps = max(avg_scores)
lowest_crps = min(avg_scores)

for score, spacing, sample_size in zip(avg_scores, full_spacings, full_sample_sizes):
    color = cmap((score - lowest_crps) / (highest_crps - lowest_crps))
    ax.scatter([spacing,], [sample_size,], color=color, zorder=-1, s=300)
    ax.scatter([spacing,], [sample_size,], color='k', zorder=-2, alpha=1, s=400)
    # ax.text(spacing, sample_size, '{:.2f}'.format(score), color='red', zorder=100, ha='left')

cbar = plt.colorbar(cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=lowest_crps, vmax=highest_crps), cmap=cmap), ax=ax, label='Score')
cbar.set_ticks([lowest_crps, highest_crps])
cbar.set_ticklabels(['Best', 'Worst'])

ax.set_xlabel('Spacing between combined survey rounds (days)')
ax.set_ylabel('Tests per round')

ax.set_yscale('log')
ax.set_xscale('log')

fig.set_tight_layout(True)

ax.set_xticks(spacings)
ax.set_yticks(sample_sizes)
ax.set_xticklabels(spacings)
ax.set_yticklabels(sample_sizes)

ax.set_xticks([], minor=True, )
ax.set_yticks([], minor=True)

plt.savefig('Figure4a.pdf')

plt.show()


In [ ]:
fig = plt.figure(figsize=(4.5, 2.5))
ax = fig.add_subplot(1, 1, 1)

for spacing in spacings:
    spacing_scores = all_scores[:, np.where(full_spacings == spacing)]
    mid = np.median(spacing_scores, axis=0)[0]
    ax.plot(sample_sizes, mid, '.-', label=spacing)
    # std = np.std(spacing_scores, axis=0)[0]
    # ax.fill_between(sample_sizes, mid - std, mid + std, alpha=0.25)

ax.legend(title='Spacing between\nrounds (days)', loc='center left', bbox_to_anchor=(1, 0.5))

ax.set_xlabel('Sample size')
ax.set_ylabel('Score')

yticks = [np.min(all_scores), np.max(all_scores)]
ax.set_yticks(yticks)
ax.set_yticklabels(['Best', 'Worst'])

ax.spines[['right', 'top']].set_visible(False)
ax.set_xlabel('Tests per round')

fig.set_tight_layout(True)

plt.savefig('Figure4b.pdf')

plt.show()

In [ ]:
all_widths = np.asarray(all_widths)
all_coverages = np.asarray(all_coverages)

In [ ]:
fig = plt.figure(figsize=(6, 3.25))

ax = fig.add_subplot(1, 1, 1)

colors = {21: 'tab:blue', 42: 'tab:orange', 84: 'tab:green', 168: 'tab:red'}

sample_sizes_to_consider = {21: (0, -1), 42: (1, None), 84: (2, None)}
scores_plotted = []

for j, spacing in enumerate(spacings):
    spacing_scores = all_scores[:, np.where(full_spacings == spacing)]
    mid = np.median(spacing_scores, axis=0)[0]

    # spacing_scores = all_scores[1, np.where(full_spacings == spacing)]
    # mid = np.mean(spacing_scores, axis=0)

    # s0, s1 = sample_sizes_to_consider[spacing]
    # mid = mid[s0:s1]
    # for i, sample_size in enumerate(sample_sizes[s0:s1]):
    for i, sample_size in enumerate(sample_sizes):
        surveys = survey_design.make_surveys(m, [spacing], [sample_size], [spacing], [sample_size], duration=7)
        effort = surveys[0].sample_size * len(surveys[0].start_dates) + surveys[1].sample_size * len(surveys[1].start_dates)

        # ax.scatter([effort,], [mid[i],], color=colors[spacing], s=50, label=spacing if i == 0 else None)
        for k in range(spacing_scores.shape[0]):
            ax.scatter([effort,], [spacing_scores[k, 0, i],], color=colors[spacing], s=25, label=spacing if k == 0 and i == 0 else None, alpha=0.5)
            scores_plotted.append(spacing_scores[k, 0, i])


ax.legend(title='Spacing between\nrounds (days)', loc='center left', bbox_to_anchor=(1, 0.5))

yticks = [np.min(scores_plotted), np.max(scores_plotted)]
ax.set_yticks(yticks)
ax.set_yticklabels(['Best', 'Worst'])

ax.spines[['right', 'top']].set_visible(False)
ax.set_xlabel('Total tests conducted')
ax.set_ylabel('Score')

ax.set_xscale('log')
# ax.set_yscale('log')

# ax.ticklabel_format(style='plain', axis='x')
plt.xticks(rotation=90, ha='center')

fig.set_tight_layout(True)

plt.savefig('Figure4b.pdf')

# ax.set_ylim(500, 17000)

plt.show()

In [ ]:
fig = plt.figure(figsize=(10, 4))
ax = fig.add_subplot(1, 1, 1)

ps = []

for j, spacing in enumerate(spacings):
    idx = np.where(full_spacings == spacing)[0]
    for i, ss in zip(idx, sample_sizes):
        p = all_fits[0][i].rw
        ps.append(p)


for i, data_i in enumerate(ps):
    lower = np.percentile(data_i, 2.5)
    upper = np.percentile(data_i, 97.5)
    ps[i] = [y for y in data_i if lower < y < upper]

v = ax.violinplot(ps)

colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']
colors_expand = np.repeat(colors, len(sample_sizes))

for i, pc in enumerate(v['bodies']):
    pc.set_facecolor(colors_expand[i])
    pc.set_edgecolor(colors_expand[i])

v['cbars'].set_edgecolor(colors_expand)
v['cmaxes'].set_edgecolor(colors_expand)
v['cmins'].set_edgecolor(colors_expand)

ax.set_xticks(np.arange(1, len(ps) + 1))
ax.set_xticklabels(list(full_sample_sizes))

ax.set_xlabel('Tests per round')
ax.set_ylabel(r'$\sigma_\text{infections}$')

ax.tick_params(axis='x', labelrotation=90)

ax.spines[['right', 'top']].set_visible(False)


from matplotlib.lines import Line2D
custom_lines = [Line2D([0], [0], color='tab:blue', lw=4, alpha=0.5),
                Line2D([0], [0], color='tab:orange', lw=4, alpha=0.5),
                Line2D([0], [0], color='tab:green', lw=4, alpha=0.5),
                Line2D([0], [0], color='tab:red', lw=4, alpha=0.5)]

ax.legend(
    custom_lines, ['21', '42', '84', '168'],
    title='Spacing between\nrounds (days)', loc='center left', bbox_to_anchor=(1, 0.5))

plt.show()

In [ ]:
surveys = survey_design.make_surveys(m, full_spacings, full_sample_sizes, full_spacings, full_sample_sizes, duration=7)

fig = plt.figure(figsize=(7, 2.75), dpi=512)

idx1 = 6
idx2 = 27
idx3 = 0
idx4 = 21

dfi = 0
df = dfs[dfi]


ax = fig.add_subplot(2, 2, 1)

fit = all_fits[dfi][idx1]

for prev_surv_start in surveys[2 * idx1].start_dates:
    if prev_surv_start not in surveys[2 * idx1 + 1].start_dates:
        l1 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='yellowgreen')
    else:
        l2 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='gray')
for sero_surv_start in surveys[2 * idx1 + 1].start_dates:
    if sero_surv_start not in surveys[2 * idx1].start_dates:
        l3 = ax.axvspan(sero_surv_start - 3.5, sero_surv_start + 3.5, alpha=0.25, color='tab:orange')
    else:
        pass

ax.plot(df['transmissions'], color='k', label='Incidence')

x = np.asarray(fit.incidence[:, :])
lower = np.percentile(x, 5, axis=0,)
upper = np.percentile(x, 95, axis=0,)
mid = np.median(x, axis=0)
tplot = np.arange(len(mid))

ax.plot(tplot, mid,)
ax.fill_between(tplot, lower, upper, alpha=0.35)
ax.spines[['right', 'top']].set_visible(False)

ax.set_ylabel('Incidence')

ax.set_title('21 Days spacing')

ax.tick_params(labelbottom=False, labelleft=True)


ax = fig.add_subplot(2, 2, 2, sharex=ax, sharey=ax)

fit = all_fits[dfi][idx2]

for prev_surv_start in surveys[2 * idx2].start_dates:
    if prev_surv_start not in surveys[2 * idx2 + 1].start_dates:
        l1 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='yellowgreen')
    else:
        l2 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='gray')
for sero_surv_start in surveys[2 * idx2 + 1].start_dates:
    if sero_surv_start not in surveys[2 * idx2].start_dates:
        l3 = ax.axvspan(sero_surv_start - 3.5, sero_surv_start + 3.5, alpha=0.25, color='tab:orange')
    else:
        pass

ax.plot(df['transmissions'], color='k', label='Incidence')

x = np.asarray(fit.incidence[:, :])
lower = np.percentile(x, 5, axis=0,)
upper = np.percentile(x, 95, axis=0,)
mid = np.median(x, axis=0)
tplot = np.arange(len(mid))

ax.plot(tplot, mid,)
ax.fill_between(tplot, lower, upper, alpha=0.35)
ax.spines[['right', 'top']].set_visible(False)

legend_ax = ax

ax.tick_params(labelbottom=False, labelleft=False)

ax.set_title('168 Days spacing')



ax = fig.add_subplot(2, 2, 3, sharex=ax, sharey=ax)

fit = all_fits[dfi][idx3]

for prev_surv_start in surveys[2 * idx3].start_dates:
    if prev_surv_start not in surveys[2 * idx3 + 1].start_dates:
        l1 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.2, color='yellowgreen')
    else:
        l2 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.2, color='gray')
for sero_surv_start in surveys[2 * idx3 + 1].start_dates:
    if sero_surv_start not in surveys[2 * idx3].start_dates:
        l3 = ax.axvspan(sero_surv_start - 3.5, sero_surv_start + 3.5, alpha=0.25, color='tab:orange')
    else:
        pass

ax.plot(df['transmissions'], color='k', label='Incidence')

x = np.asarray(fit.incidence[:, :])
lower = np.percentile(x, 5, axis=0,)
upper = np.percentile(x, 95, axis=0,)
mid = np.median(x, axis=0)
tplot = np.arange(len(mid))

ax.plot(tplot, mid,)
ax.fill_between(tplot, lower, upper, alpha=0.35)
ax.spines[['right', 'top']].set_visible(False)


ax.set_ylabel('Incidence')
ax.set_xlabel('Time (days)')

ax.set_title('1000 Tests\nper round', x=-0.5, y=0.4)



ax = fig.add_subplot(2, 2, 4, sharex=ax, sharey=ax)

fit = all_fits[dfi][idx4]

for prev_surv_start in surveys[2 * idx4].start_dates:
    if prev_surv_start not in surveys[2 * idx4 + 1].start_dates:
        l1 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='yellowgreen')
    else:
        l2 = ax.axvspan(prev_surv_start - 3.5, prev_surv_start + 3.5, alpha=0.25, color='gray')
for sero_surv_start in surveys[2 * idx4 + 1].start_dates:
    if sero_surv_start not in surveys[2 * idx4].start_dates:
        l3 = ax.axvspan(sero_surv_start - 3.5, sero_surv_start + 3.5, alpha=0.25, color='tab:orange')
    else:
        pass

l6, = ax.plot(df['transmissions'], color='k', label='Incidence')

x = np.asarray(fit.incidence[:, :])
lower = np.percentile(x, 5, axis=0,)
upper = np.percentile(x, 95, axis=0,)
mid = np.median(x, axis=0)
tplot = np.arange(len(mid))

l4, = ax.plot(tplot, mid,)
l5 = ax.fill_between(tplot, lower, upper, alpha=0.35)
ax.spines[['right', 'top']].set_visible(False)
ax.set_xlabel('Time (days)')

ax.set_title('64000 Tests\nper round', x=-1.7, y=1.6)

legend_ax.legend((l2, (l4, l5,), l6,), ['Combined survey', 'Posterior', 'Incidence'], loc='center left', bbox_to_anchor=(1, 0.5))

fig.set_tight_layout(True)

ax.tick_params(labelbottom=True, labelleft=False)

plt.savefig('Figure4c.pdf')

plt.show()